In [17]:
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.evaluation import EvaluateRetrieval
from beir import util, LoggingHandler

import logging
import pathlib, os

In [18]:
#### Just some code to print debug information to stdout
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

In [34]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='cuda')

In [25]:
!export CUDA_LAUNCH_BLOCKING=1

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
input_path = "../datasets/Bharat_NanoMSMARCO/qrels/test.tsv"
output_path = "../datasets/Bharat_NanoMSMARCO/qrels/test_fixed.tsv"

with (
    open(input_path, "r", encoding="utf-8") as fin,
    open(output_path, "w", encoding="utf-8") as fout,
):
    for line in fin:
        parts = line.strip().split()
        if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
            fout.write(f"{parts[0]}\t{parts[1]}\t1\n")
# Now use test_fixed.tsv as your qrels

In [27]:
########## 1. Set Paths ##########
data_path = "../datasets/Bharat_NanoMSMARCO"  # Replace with your path

########## 2. Load Data ##########
corpus, queries, qrels = GenericDataLoader(
    corpus_file=f"{data_path}/corpus.jsonl",
    query_file=f"{data_path}/queries.jsonl",
    qrels_file=f"{data_path}/qrels/test.tsv",
).load_custom()

for doc_id, doc in corpus.items():
    if doc.get("title") is None:
        doc["title"] = ""
    if doc.get("text") is None:
        doc["text"] = ""

2025-08-05 00:09:40 - Loading Corpus...


100%|██████████| 5043/5043 [00:00<00:00, 179787.97it/s]

2025-08-05 00:09:40 - Loaded 5043 Documents.
2025-08-05 00:09:40 - Doc Example: {'text': 'n (ബ്രിട്ടീഷ് ഭാഷയിൽ പ്രാദേശിക പോലീസ് സേനയുടെ കാര്യക്ഷമതയ്ക്ക് ഉത്തരവാദികളായ ജില്ലാ ബോർഡ് കൌൺസിലിന്റെ പ്രതിനിധികളും മജിസ്ട്രേറ്റുകളും ചേർന്ന ഒരു പ്രാദേശിക ഭരണ സമിതി. ഇംഗ്ലീഷ് കോളിൻസ് നിഘണ്ടു-ഇംഗ്ലീഷ് നിർവചനം & തിസറസ് &nbsp.', 'title': None}
2025-08-05 00:09:40 - Loading Queries...
2025-08-05 00:09:40 - Loaded 49 Queries.
2025-08-05 00:09:40 - Query Example: ആരോഗ്യ പരിരക്ഷയിൽ എന്താണ് ശരിയെന്ന്


In [35]:
########## 3. Load mE5 Model ##########
# Choose either "intfloat/multilingual-e5-base" or "intfloat/multilingual-e5-large"
model_name = "intfloat/multilingual-e5-base"  # or "intfloat/multilingual-e5-large"
sentence_model = models.SentenceBERT(model_name, device=device.type)
dres = DRES(sentence_model, batch_size=12)

2025-08-05 00:19:35 - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


RuntimeError: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [31]:
########## 4. Retrieve ##########
retriever = EvaluateRetrieval(dres, score_function="cos_sim")
results = retriever.retrieve(corpus, queries)

2025-08-05 00:11:32 - Encoding Queries...


Batches: 100%|██████████| 13/13 [00:00<00:00, 24.69it/s]


2025-08-05 00:11:33 - Sorting Corpus by document length (Longest first)...
2025-08-05 00:11:33 - Encoding Corpus in batches... Warning: This might take a while!
2025-08-05 00:11:33 - Scoring Function: Cosine Similarity (cos_sim)
2025-08-05 00:11:33 - Encoding Batch 1/1...


Batches: 100%|██████████| 1261/1261 [05:23<00:00,  3.90it/s]


In [32]:
########## 5. Evaluate ##########
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, retriever.k_values)
print("NDCG:", ndcg)
print("MAP:", _map)
print("Recall:", recall)
print("Precision:", precision)

2025-08-05 00:16:59 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-08-05 00:16:59 - 

2025-08-05 00:16:59 - NDCG@1: 0.2245
2025-08-05 00:16:59 - NDCG@3: 0.3528
2025-08-05 00:16:59 - NDCG@5: 0.3879
2025-08-05 00:16:59 - NDCG@10: 0.4134
2025-08-05 00:16:59 - NDCG@100: 0.4861
2025-08-05 00:16:59 - NDCG@1000: 0.4937
2025-08-05 00:16:59 - 

2025-08-05 00:16:59 - MAP@1: 0.2245
2025-08-05 00:16:59 - MAP@3: 0.3197
2025-08-05 00:16:59 - MAP@5: 0.3401
2025-08-05 00:16:59 - MAP@10: 0.3502
2025-08-05 00:16:59 - MAP@100: 0.3659
2025-08-05 00:16:59 - MAP@1000: 0.3661
2025-08-05 00:16:59 - 

2025-08-05 00:16:59 - Recall@1: 0.2245
2025-08-05 00:16:59 - Recall@3: 0.4490
2025-08-05 00:16:59 - Recall@5: 0.5306
2025-08-05 00:16:59 - Recall@10: 0.6122
2025-08-05 00:16:59 - Recall@100: 0.9388
2025-08-05 00:16:59 - Recall@1000: 1.0000
2025-08-05 00:16:59 - 

2025-08-05 00:16:59 - P@1: 0.2245
2025-08-05 00:16:59